In [ ]:
import os
import csv
from pathlib import Path

import numpy as np
import soundfile as sf


ROOT = Path(r"./TIMIT")
TIMIT_ROOT = ROOT / "data"             # it should contain TRAIN/TEST
OUT_MANIFEST_DIR = ROOT / "manifests"
OUT_LABEL_DIR = ROOT / "frame_labels"

OUT_MANIFEST_DIR.mkdir(exist_ok=True)
OUT_LABEL_DIR.mkdir(exist_ok=True)

SAMPLE_RATE = 16000
FRAME_LEN_MS = 25.0
FRAME_HOP_MS = 10.0

FRAME_LEN = int(SAMPLE_RATE * FRAME_LEN_MS / 1000.0)   
FRAME_HOP = int(SAMPLE_RATE * FRAME_HOP_MS / 1000.0)   


NON_SPEECH_PHONES = {"sil", "sp", "spn", "pau", "h#", "epi"}



def list_timit_utterances(split="TRAIN"):
    """
    Iterate over all utterances in data/TRAIN or data/TEST.
    Returns tuples (wav_path, phn_path, dialect, speaker).
    """
    split_dir = TIMIT_ROOT / split
    for dr in sorted(split_dir.glob("DR*")):
        dialect = dr.name  
        for spk in sorted(dr.iterdir()):
            if not spk.is_dir():
                continue
            speaker = spk.name
            for phn_file in sorted(spk.glob("*.PHN")):
                base = phn_file.stem  
                wav_orig = spk / f"{base}.WAV.wav"
                if not wav_orig.exists():
                    
                    wav_orig = spk / f"{base}.WAV"
                if not wav_orig.exists():
                    continue
                yield wav_orig, phn_file, dialect, speaker

def load_phn_intervals(phn_path):
    """
    Load TIMIT .PHN file.
    Returns list of (start_sample, end_sample, phone_string).
    """
    intervals = []
    with open(phn_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 3:
                continue
            start, end, phone = parts
            intervals.append((int(start), int(end), phone.lower()))
    return intervals

def frames_for_len(n_samples):
    """Number of frames given signal length, frame length and hop."""
    if n_samples <= FRAME_LEN:
        return 1
    return 1 + int((n_samples - FRAME_LEN) / FRAME_HOP)

def make_frame_labels(phn_intervals, num_frames):
    """
    Create frame-level VAD labels (0/1) from phone intervals.
    A frame is speech if its center falls inside any speech phone interval.
    """
    labels = np.zeros(num_frames, dtype=np.int64)
    
    speech_intervals = []
    for start, end, phone in phn_intervals:
        if phone in NON_SPEECH_PHONES:
            continue
        speech_intervals.append((start, end))
    if not speech_intervals:
        return labels  

    for i in range(num_frames):
       
        center = i * FRAME_HOP + FRAME_LEN // 2
        for s, e in speech_intervals:
            if s <= center < e:
                labels[i] = 1
                break
    return labels



def process_split(split):
    """
    Build a CSV manifest and per-utterance frame-label .npy files
    for TRAIN or TEST.
    """
    manifest_path = OUT_MANIFEST_DIR / f"{split.lower()}_manifest.csv"
    with open(manifest_path, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
       
        writer.writerow([
            "utt_id", "wav_path", "split", "dialect", "speaker",
            "n_samples", "duration_sec", "n_frames", "label_path"
        ])

        for wav_path, phn_path, dialect, speaker in list_timit_utterances(split):
            
            audio, sr = sf.read(str(wav_path))
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            if sr != SAMPLE_RATE:
                
                raise ValueError(f"Unexpected sample rate {sr} in {wav_path}")

            n_samples = len(audio)
            duration = n_samples / SAMPLE_RATE
            n_frames = frames_for_len(n_samples)

            
            phn_intervals = load_phn_intervals(phn_path)
            labels = make_frame_labels(phn_intervals, n_frames)

            
            utt_id = f"{dialect}_{speaker}_{wav_path.stem}"
            label_path = OUT_LABEL_DIR / f"{utt_id}_labels.npy"
            np.save(label_path, labels)

            writer.writerow([
                utt_id,
                str(wav_path),
                split,
                dialect,
                speaker,
                n_samples,
                f"{duration:.4f}",
                n_frames,
                str(label_path)
            ])

    print(f"Wrote manifest: {manifest_path}")

if __name__ == "__main__":
    process_split("TRAIN")
    process_split("TEST")


In [ ]:
import os
import csv
import random
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa


ROOT = Path(r"./TIMIT")

MANIFEST_DIR = ROOT / "manifests"
LABEL_DIR = ROOT / "frame_labels"

NOISY_AUDIO_ROOT = ROOT / "noisy_audio"
NOISY_MANIFEST_DIR = ROOT / "noisy_manifests"
NOISY_AUDIO_ROOT.mkdir(exist_ok=True)
NOISY_MANIFEST_DIR.mkdir(exist_ok=True)

SAMPLE_RATE = 16000
TARGET_SNRS = [-5, 0, 5]  


TRAIN_NOISE_ROOT = ROOT / "TRAIN_NOISE"  
TEST_NOISE_ROOT = ROOT / "TEST_NOISE"         


TRAIN_NOISE_TYPES = [
    "BABBLE",
    "DKITCHEN",
    "DWASHING",
    "OOFFICE",
    "PCAFETER",
    "STRAFFIC",
    "TBUS",
]

TRAIN_NOISE_FILE_MAP = {
    "BABBLE": "babble.wav",
    "DKITCHEN": "ch01.wav",
    "DWASHING": "ch01.wav",
    "OOFFICE": "ch01.wav",
    "PCAFETER": "ch01.wav",
    "STRAFFIC": "ch01.wav",
    "TBUS": "ch01.wav",
}




def read_wav_mono_resampled(path, target_sr=SAMPLE_RATE):
    """
    Load audio, convert to mono, resample to target_sr if needed.
    """
    audio, sr = sf.read(str(path))
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != target_sr:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        sr = target_sr
    return audio


def ensure_length(noise, target_len):
    """
    Loop or random-crop noise to get exactly target_len samples.
    """
    if len(noise) >= target_len:
        start = random.randint(0, len(noise) - target_len)
        return noise[start:start + target_len]
    else:
        reps = int(np.ceil(target_len / len(noise)))
        tiled = np.tile(noise, reps)
        return tiled[:target_len]


def mix_at_snr(clean, noise, snr_db):
    """
    Mix clean + noise at a target SNR (dB) based on average power.
    """
    eps = 1e-12
    clean_power = np.mean(clean ** 2) + eps
    noise_power = np.mean(noise ** 2) + eps

    snr_linear = 10 ** (snr_db / 10.0)
    target_noise_power = clean_power / snr_linear

    scale = np.sqrt(target_noise_power / noise_power)
    noise_scaled = noise * scale

    noisy = clean + noise_scaled

    
    peak = np.max(np.abs(noisy))
    if peak > 0.99:
        noisy = noisy / peak * 0.99

    return noisy


def load_noise_for_type(noise_root, noise_type, expected_file=None):
    """
    Load a noise signal given a root folder and a subfolder (noise_type).
    If expected_file is provided, use that; otherwise, use the first .wav.
    """
    folder = noise_root / noise_type
    if expected_file is not None:
        path = folder / expected_file
        if not path.exists():
            raise FileNotFoundError(f"Noise file not found: {path}")
        return read_wav_mono_resampled(path)
    else:
        wavs = sorted(folder.glob("*.wav"))
        if not wavs:
            raise FileNotFoundError(f"No .wav files in {folder}")
        return read_wav_mono_resampled(wavs[0])


def find_test_noise_types():
    """
    List noise subfolders in TEST/ (held-out noises).
    """
    return [p.name for p in sorted(TEST_NOISE_ROOT.iterdir()) if p.is_dir()]




def make_noisy_split(split):
    """
    Create noisy mixtures for a given split ('train' or 'test').

    For 'train':
      - uses TRAIN_NOISE_ROOT and TRAIN_NOISE_TYPES
    For 'test':
      - uses TEST_NOISE_ROOT and all its subfolders as unseen noises
    """
    in_manifest = MANIFEST_DIR / f"{split}_manifest.csv"
    out_manifest = NOISY_MANIFEST_DIR / f"{split}_noisy_manifest.csv"
    out_audio_dir = NOISY_AUDIO_ROOT / split
    out_audio_dir.mkdir(exist_ok=True)

    if split == "train":
        noise_root = TRAIN_NOISE_ROOT
        noise_types = TRAIN_NOISE_TYPES
        noise_file_map = TRAIN_NOISE_FILE_MAP
    else:  
        noise_root = TEST_NOISE_ROOT
        noise_types = find_test_noise_types()
        noise_file_map = {nt: "ch01.wav" for nt in noise_types}

    noise_signals = {}
    for nt in noise_types:
        fname = noise_file_map.get(nt)
        noise_signals[nt] = load_noise_for_type(noise_root, nt, fname)
        print(f"Loaded noise '{nt}' from {noise_root / nt / (fname or '')}")

    with open(in_manifest, "r", encoding="utf-8") as f_in, \
         open(out_manifest, "w", newline="", encoding="utf-8") as f_out:

        reader = csv.DictReader(f_in)
        fieldnames = list(reader.fieldnames) + ["noise_type", "snr_db", "noisy_wav_path"]
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            wav_path = Path(row["wav_path"])
            utt_id = row["utt_id"]

            
            clean, sr = sf.read(str(wav_path))
            if clean.ndim > 1:
                clean = clean.mean(axis=1)
            if sr != SAMPLE_RATE:
                clean = librosa.resample(clean, orig_sr=sr, target_sr=SAMPLE_RATE)
                sr = SAMPLE_RATE
            n_samples = len(clean)

            for snr in TARGET_SNRS:
               
                noise_type = random.choice(noise_types)
                noise = noise_signals[noise_type]
                noise_segment = ensure_length(noise, n_samples)

                noisy = mix_at_snr(clean, noise_segment, snr)

               
                noisy_rel_dir = out_audio_dir / f"{noise_type}_SNR{snr}"
                noisy_rel_dir.mkdir(exist_ok=True)
                noisy_fname = f"{utt_id}_SNR{snr}_{noise_type}.wav"
                noisy_path = noisy_rel_dir / noisy_fname
                sf.write(str(noisy_path), noisy, SAMPLE_RATE)

                
                out_row = dict(row)
                out_row["noise_type"] = noise_type
                out_row["snr_db"] = str(snr)
                out_row["noisy_wav_path"] = str(noisy_path)
                writer.writerow(out_row)

    print(f"Wrote noisy manifest: {out_manifest}")


if __name__ == "__main__":
    random.seed(42)  
    make_noisy_split("train")
    make_noisy_split("test")


In [ ]:
import os
import csv
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa
import scipy.signal as signal
from tqdm import tqdm


ROOT = Path(r"./TIMIT")

SAMPLE_RATE = 16000
FRAME_HOP = 160        
MAX_FRAMES = 1500      


NOISY_MANIFEST_DIR = ROOT / "noisy_manifests"
TRAIN_NOISY_MANIFEST = NOISY_MANIFEST_DIR / "train_noisy_manifest.csv"
TEST_NOISY_MANIFEST = NOISY_MANIFEST_DIR / "test_noisy_manifest.csv"


LABEL_DIR = ROOT / "frame_labels"


WIDEBAND_FEATURE_ROOT = ROOT / "processed_features_wideband"
INDUSTRY_FEATURE_ROOT = ROOT / "industry_features"

MANUFACTURERS = ["cochlear", "medel", "ab"]
SNRS = ["-5", "0", "5"]  



def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)




class CI_Strategy_Emulator:
    def __init__(self, strategy='cochlear', fs=16000, hop_length=160):
        self.strategy = strategy.lower()
        self.fs = fs
        self.nyquist = fs / 2.0
        self.hop_length = hop_length

        safe_max_freq = self.nyquist - 50  

        if self.strategy == 'cochlear':
            self.num_channels = 22
            self.fft_size = 256
            self.hop_size = 160  
           
            self.bin_map = self._create_fft_bin_map(188, 7938)
        elif self.strategy == 'medel':
            self.num_channels = 12
            self.edges = self._get_medel_edges(self.num_channels, 70, safe_max_freq)
            self.sos_filters = self._create_filterbank(self.edges)
            self.lp_filter = self._create_lpf_rectifier()
        elif self.strategy == 'ab':
            self.num_channels = 16
            self.edges = np.logspace(np.log10(250), np.log10(safe_max_freq),
                                     self.num_channels + 1)
            self.sos_filters = self._create_filterbank(self.edges)
            self.lp_filter = self._create_lpf_rectifier()
        else:
            raise ValueError(f"Unknown strategy: {self.strategy}")

    
    def _create_fft_bin_map(self, low_f, high_f):
        bin_freqs = np.fft.rfftfreq(self.fft_size, d=1 / self.fs)
        channel_boundaries = np.logspace(np.log10(low_f), np.log10(high_f),
                                         self.num_channels + 1)
        bin_map = np.zeros((self.num_channels, len(bin_freqs)))
        for ch in range(self.num_channels):
            low, high = channel_boundaries[ch], channel_boundaries[ch + 1]
            idx = np.where((bin_freqs >= low) & (bin_freqs < high))[0]
            if len(idx) > 0:
                bin_map[ch, idx] = 1.0
        return bin_map

    def _process_ace_fft(self, audio):
        
        f, t, Zxx = signal.stft(audio, fs=self.fs, window='hann',
                                nperseg=self.fft_size,
                                noverlap=self.fft_size - self.hop_size)
        magnitude = np.abs(Zxx) 
        envelopes = self.bin_map @ magnitude  
        m = np.max(envelopes)
        if m > 1e-9:
            envelopes /= m
        return envelopes  

   
    def _get_medel_edges(self, n_channels, low, high):
        split = 400
        n_lin = 2
        lin_edges = np.linspace(low, split, n_lin + 1)
        log_edges = np.logspace(np.log10(split), np.log10(high),
                                (n_channels - n_lin) + 1)
        return np.concatenate((lin_edges[:-1], log_edges))

    def _create_filterbank(self, edges):
        sos_list = []
        for i in range(len(edges) - 1):
            low = max(edges[i], 1.0)
            high = min(edges[i + 1], self.nyquist - 1.0)
            sos = signal.butter(6, [low, high], btype='band', fs=self.fs, output='sos')
            sos_list.append(sos)
        return sos_list

    def _create_lpf_rectifier(self):
        return signal.butter(2, 400, btype='low', fs=self.fs, output='sos')

    def _process_cis_filterbank(self, audio):
        
        envelopes = np.zeros((self.num_channels, len(audio)))
        for i, sos in enumerate(self.sos_filters):
            filtered = signal.sosfilt(sos, audio)
            rectified = np.abs(filtered)
            env = signal.sosfilt(self.lp_filter, rectified)
            envelopes[i, :] = env
        m = np.max(envelopes)
        if m > 1e-9:
            envelopes /= m
        
        return envelopes[:, ::self.hop_length]  

    def process(self, audio):
        if self.strategy == 'cochlear':
            return self._process_ace_fft(audio)
        else:
            return self._process_cis_filterbank(audio)




def load_audio_16k(path: Path):
    audio, sr = sf.read(str(path))
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    return audio


def pad_truncate_frames(arr, max_frames):
    """
    Arr: shape [T, ...] or [C, T], pad/truncate along time dim to max_frames.
    Returns array with same leading dims but T=max_frames.
    """
    if arr.ndim == 1:
        T = arr.shape[0]
        if T < max_frames:
            pad = max_frames - T
            return np.pad(arr, (0, pad), mode='constant')
        else:
            return arr[:max_frames]
    elif arr.ndim == 2:
       
        raise ValueError("pad_truncate_frames is for 1D; use specific logic for 2D.")


def pad_truncate_time_axis(feat, labels, max_frames):
    """
    feat: [C, T]  labels: [T]
    Align labels to feat, then pad/truncate both to max_frames.
    """
    C, T_feat = feat.shape
    T_lab = len(labels)

    if T_lab > T_feat:
        labels = labels[:T_feat]
        T_lab = T_feat
    elif T_lab < T_feat:
        labels = np.pad(labels, (0, T_feat - T_lab), mode='constant')
        T_lab = T_feat

   
    if T_feat < max_frames:
        pad = max_frames - T_feat
        feat2 = np.pad(feat, ((0, 0), (0, pad)), mode='constant')
        labels2 = np.pad(labels, (0, pad), mode='constant')
    else:
        feat2 = feat[:, :max_frames]
        labels2 = labels[:max_frames]

    return feat2, labels2




def extract_wideband_features(manifest_csv: Path, split: str):
    """
    For each noisy utterance, save:
      - audio framed implicitly (raw audio will be used directly by wideband TCN)
      - frame labels at 10 ms, padded/truncated to MAX_FRAMES
    """
    print(f"\n=== Wideband feature prep for {split} ===")
    out_audio_root = WIDEBAND_FEATURE_ROOT / f"noisy_{split}" / "audio"
    out_label_root = WIDEBAND_FEATURE_ROOT / f"noisy_{split}" / "labels"
    ensure_dir(out_audio_root)
    ensure_dir(out_label_root)

    with open(manifest_csv, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in tqdm(reader):
            utt_id = row["utt_id"]
            noisy_path = Path(row["noisy_wav_path"])
            label_path = LABEL_DIR / f"{utt_id}_labels.npy"

            if not noisy_path.exists():
                continue
            if not label_path.exists():
                continue

            audio = load_audio_16k(noisy_path)
            labels = np.load(label_path)  
            if len(labels) > MAX_FRAMES:
                labels = labels[:MAX_FRAMES]

            out_audio_path = out_audio_root / f"{utt_id}.wav"
            out_label_path = out_label_root / f"{utt_id}.npy"

            sf.write(str(out_audio_path), audio, SAMPLE_RATE)
            np.save(out_label_path, labels)




def extract_industry_features(manifest_csv: Path, split: str):
    """
    For each noisy utterance, generate manufacturer-specific envelope features:
      - cochlear: [22, T]
      - medel: [12, T]
      - ab:     [16, T]
    and aligned labels, all padded/truncated to MAX_FRAMES along time axis.
    """
    print(f"\nIndustry CI-like feature prep for {split} ")
    with open(manifest_csv, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    # Pre-instantiate simulators
    simulators = {
        "cochlear": CI_Strategy_Emulator("cochlear", fs=SAMPLE_RATE, hop_length=FRAME_HOP),
        "medel": CI_Strategy_Emulator("medel", fs=SAMPLE_RATE, hop_length=FRAME_HOP),
        "ab": CI_Strategy_Emulator("ab", fs=SAMPLE_RATE, hop_length=FRAME_HOP),
    }

    for manufacturer in MANUFACTURERS:
        print(f"\n {manufacturer.upper()} ({split})")
        for row in tqdm(rows, desc=f"{manufacturer}-{split}"):
            utt_id = row["utt_id"]
            noisy_path = Path(row["noisy_wav_path"])
            label_path = LABEL_DIR / f"{utt_id}_labels.npy"

            if not noisy_path.exists():
                continue
            if not label_path.exists():
                continue

            audio = load_audio_16k(noisy_path)
            labels = np.load(label_path)  

           
            envs = simulators[manufacturer].process(audio)

           
            feat, lab = pad_truncate_time_axis(envs, labels, MAX_FRAMES) 

           
            out_feat_dir = INDUSTRY_FEATURE_ROOT / manufacturer / f"noisy_{split}" / "features"
            out_lbl_dir = INDUSTRY_FEATURE_ROOT / manufacturer / f"noisy_{split}" / "labels"
            ensure_dir(out_feat_dir)
            ensure_dir(out_lbl_dir)

            base_name = f"{utt_id}.npy"
            np.save(out_feat_dir / base_name, feat.astype(np.float32))
            np.save(out_lbl_dir / base_name, lab.astype(np.int8))




def main():
   
    extract_wideband_features(TRAIN_NOISY_MANIFEST, split="train")
    extract_wideband_features(TEST_NOISY_MANIFEST, split="test")

    
    extract_industry_features(TRAIN_NOISY_MANIFEST, split="train")
    extract_industry_features(TEST_NOISY_MANIFEST, split="test")

    print("\nwideband and CI-like features extracted.")


if __name__ == "__main__":
    main()


In [ ]:
import os
import glob
from pathlib import Path

import numpy as np
import soundfile as sf
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import torch_directml

# ========= CONFIG =========
ROOT = Path(r"./TIMIT")

FEATURE_ROOT = ROOT / "processed_features_wideband"
TRAIN_AUDIO_DIR = FEATURE_ROOT / "noisy_train" / "audio"
TRAIN_LABEL_DIR = FEATURE_ROOT / "noisy_train" / "labels"

SNRS = ["-5", "0", "5"]  

BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-3

FRAME_HOP = 160       
MAX_FRAMES = 1500       
MAX_AUDIO_LEN = MAX_FRAMES * FRAME_HOP

DEVICE = torch_directml.device()
print(f"--- Training Wideband VAD on {DEVICE} ---")



class WidebandVADDataset(Dataset):
    def __init__(self, utt_ids):
        self.utt_ids = utt_ids
        self.max_frames = MAX_FRAMES
        self.hop_length = FRAME_HOP
        self.max_audio_len = MAX_AUDIO_LEN

    def __len__(self):
        return len(self.utt_ids)

    def __getitem__(self, idx):
        utt_id = self.utt_ids[idx]
        audio_path = TRAIN_AUDIO_DIR / f"{utt_id}.wav"
        label_path = TRAIN_LABEL_DIR / f"{utt_id}.npy"

       
        y = np.load(label_path)  
        if len(y) < self.max_frames:
            y = np.pad(y, (0, self.max_frames - len(y)), mode='constant')
        else:
            y = y[:self.max_frames]

        x_audio, sr = sf.read(str(audio_path))
        if x_audio.ndim > 1:
            x_audio = x_audio.mean(axis=1)
        if sr != 16000:
            raise ValueError(f"Expected 16 kHz, got {sr} in {audio_path}")
        if len(x_audio) < self.max_audio_len:
            x_audio = np.pad(x_audio, (0, self.max_audio_len - len(x_audio)), mode='constant')
        else:
            x_audio = x_audio[:self.max_audio_len]

        X = torch.tensor(x_audio, dtype=torch.float32).unsqueeze(0)  
        y = torch.tensor(y, dtype=torch.float32)                     

        return X, y




class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size,
                               padding=self.pad, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size,
                               padding=self.pad, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = out[:, :, :-self.pad]
        out = self.relu(self.bn1(out))
        out = self.dropout(out)
        out = self.conv2(out)
        out = out[:, :, :-self.pad]
        out = self.bn2(out)
        if self.downsample is not None:
            residual = self.downsample(x)
        return self.relu(out + residual)


class WidebandLongTCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # downsampling 240k samples to 1500 frames (10 ms)
            nn.Conv1d(1, 24, kernel_size=160, stride=160),
            ResidualBlock(24, 24, 3, 1),
            ResidualBlock(24, 24, 3, 2),
            ResidualBlock(24, 24, 3, 4),
            ResidualBlock(24, 24, 3, 8),
            ResidualBlock(24, 24, 3, 16),
            ResidualBlock(24, 24, 3, 32),
            ResidualBlock(24, 24, 3, 64),
            ResidualBlock(24, 24, 3, 128),
            ResidualBlock(24, 24, 3, 256),
            ResidualBlock(24, 24, 3, 512),            
            nn.Conv1d(24, 1, 1),
        )

    def forward(self, x):
        
        return self.net(x).squeeze(1)




def main():
    
    label_files = sorted(TRAIN_LABEL_DIR.glob("*.npy"))
    utt_ids = [p.stem for p in label_files]
    print(f"Found {len(utt_ids)} training utterances.")

    train_ids, val_ids = train_test_split(utt_ids, test_size=0.2, random_state=42)

    train_loader = DataLoader(WidebandVADDataset(train_ids),
                              batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(WidebandVADDataset(val_ids),
                            batch_size=BATCH_SIZE, shuffle=False)

    model = WidebandLongTCN().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

    best_val_loss = float('inf')

    for epoch in range(1, EPOCHS + 1):
        
        model.train()
        train_loss = 0.0
        train_acc = 0.0

        for X, y in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]"):
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(X)  # [B, T]
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            train_acc += (preds == y).float().mean().item() * X.size(0)

        
        model.eval()
        val_loss = 0.0
        val_acc = 0.0
        with torch.no_grad():
            for X, y in DataLoader(WidebandVADDataset(val_ids),
                                   batch_size=BATCH_SIZE, shuffle=False):
                X, y = X.to(DEVICE), y.to(DEVICE)
                outputs = model(X)
                loss = criterion(outputs, y)
                val_loss += loss.item() * X.size(0)
                preds = (torch.sigmoid(outputs) > 0.5).float()
                val_acc += (preds == y).float().mean().item() * X.size(0)

        t_loss = train_loss / len(train_ids)
        t_acc = train_acc / len(train_ids) * 100
        v_loss = val_loss / len(val_ids)
        v_acc = val_acc / len(val_ids) * 100

        print(f"Epoch {epoch:02d} | Train Loss: {t_loss:.4f}, Acc: {t_acc:.2f}% | "
              f"Val Loss: {v_loss:.4f}, Acc: {v_acc:.2f}%")

        scheduler.step(v_loss)

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save(model.state_dict(), ROOT / "best_wideband_tcn_512.pth")
            print("New Best Wideband Model Saved!")

    print(" Wideband training complete.")


if __name__ == "__main__":
    main()


In [ ]:
import os
import glob
from pathlib import Path

import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import torch_directml


ROOT = Path(r"./TIMIT")

DATA_ROOT = ROOT / "industry_features"
MANUFACTURERS = ["cochlear", "medel", "ab"]

BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-3
MAX_FRAMES = 1500

DEVICE = torch_directml.device()
print(f"Training CI-like VAD on {DEVICE}")



class IndustryDataset(Dataset):
    def __init__(self, file_list, target_len=MAX_FRAMES):
        self.file_list = file_list
        self.max_len = target_len

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        feat_path = self.file_list[idx]
        
        lbl_path = feat_path.replace(os.sep + "features" + os.sep,
                                     os.sep + "labels" + os.sep)

        try:
            X = np.load(feat_path)   
            y = np.load(lbl_path)    
        except Exception:
            return torch.zeros(1, self.max_len), torch.zeros(self.max_len)

        C, T_feat = X.shape
        T_lab = len(y)

        
        if T_lab > T_feat:
            y = y[:T_feat]
            T_lab = T_feat
        elif T_lab < T_feat:
            y = np.pad(y, (0, T_feat - T_lab), mode='constant')
            T_lab = T_feat

       
        if T_feat < self.max_len:
            pad = self.max_len - T_feat
            X = np.pad(X, ((0, 0), (0, pad)), mode='constant')
            y = np.pad(y, (0, pad), mode='constant')
        else:
            X = X[:, :self.max_len]
            y = y[:self.max_len]

        X = torch.tensor(X, dtype=torch.float32)   
        y = torch.tensor(y, dtype=torch.float32)   

        return X, y




class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size,
                               padding=self.pad, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size,
                               padding=self.pad, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x):
        res = x
        out = self.conv1(x)
        out = out[:, :, :-self.pad]
        out = self.relu(self.bn1(out))
        out = self.conv2(out)
        out = out[:, :, :-self.pad]
        out = self.bn2(out)
        if self.downsample is not None:
            res = self.downsample(x)
        return self.relu(out + res)


class IndustryTCN(nn.Module):
    def __init__(self, input_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(input_channels, 24, 1),
            ResidualBlock(24, 24, 3, 1),
            ResidualBlock(24, 24, 3, 2),
            ResidualBlock(24, 24, 3, 4),
            ResidualBlock(24, 24, 3, 8),
            ResidualBlock(24, 24, 3, 16),
            ResidualBlock(24, 24, 3, 32),
            ResidualBlock(24, 24, 3, 64),
            ResidualBlock(24, 24, 3, 128),
            ResidualBlock(24, 24, 3, 256),
            ResidualBlock(24, 24, 3, 512),
            nn.Conv1d(24, 1, 1)
        )

    def forward(self, x):
        
        return self.net(x).squeeze(1)




def train_one_manufacturer(brand: str):
    print(f"\nTraining for manufacturer: {brand}")

    if "cochlear" in brand:
        ch_in = 22
    elif "ab" in brand:
        ch_in = 16
    elif "medel" in brand:
        ch_in = 12
    else:
        return

    brand_dir = DATA_ROOT / brand
    
    feat_dir = brand_dir / "noisy_train" / "features"
    all_files = sorted(glob.glob(str(feat_dir / "*.npy")))

    if len(all_files) == 0:
        print(f"No feature files found for {brand} in {feat_dir}")
        return

    train_files, val_files = train_test_split(all_files, test_size=0.2, random_state=42)
    train_loader = DataLoader(IndustryDataset(train_files), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(IndustryDataset(val_files), batch_size=BATCH_SIZE, shuffle=False)

    print(f" Loaded {len(train_files)} training files, {len(val_files)} validation files.")

    model = IndustryTCN(input_channels=ch_in).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3)

    best_loss = float('inf')
    save_name = ROOT / f"best_{brand}_tcn_512.pth"

    for epoch in range(1, EPOCHS + 1):
        
        model.train()
        total_train_loss = 0.0
        train_correct = 0
        train_total = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch} [{brand}]", leave=True)
        for X, y in pbar:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(X)         
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            preds = (torch.sigmoid(out) > 0.5).float()
            train_correct += (preds == y).sum().item()
            train_total += y.numel()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_train_loss = total_train_loss / len(train_loader)
        avg_train_acc = 100.0 * train_correct / train_total

       
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                out = model(X)
                l = criterion(out, y)
                val_loss += l.item()
                preds = (torch.sigmoid(out) > 0.5).float()
                val_correct += (preds == y).sum().item()
                val_total += y.numel()

        avg_val_loss = val_loss / len(val_loader)
        avg_val_acc = 100.0 * val_correct / val_total
        scheduler.step(avg_val_loss)

        tqdm.write(
            f" Ep {epoch}: Train Loss: {avg_train_loss:.4f} "
            f"(Acc: {avg_train_acc:.2f}%) | "
            f"Val Loss: {avg_val_loss:.4f} (Acc: {avg_val_acc:.2f}%)"
        )

        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            torch.save(model.state_dict(), save_name)
            tqdm.write(" Saved New Best Model")

    print(f"Finished {brand}. Best model: {save_name}")


def main():
    if not DATA_ROOT.exists():
        print(f"DATA_ROOT '{DATA_ROOT}' does not exist.")
        return

    for brand in MANUFACTURERS:
        train_one_manufacturer(brand)


if __name__ == "__main__":
    main()


In [ ]:
import os
import csv
from pathlib import Path
from collections import defaultdict

import numpy as np
import soundfile as sf
from tqdm import tqdm

import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

import torch_directml

ROOT = Path(r"./TIMIT")

LABEL_DIR = ROOT / "processed_features_wideband" / "noisy_test" / "labels"

WIDEBAND_AUDIO_DIR = ROOT / "processed_features_wideband" / "noisy_test" / "audio"

INDUSTRY_FEATURE_ROOT = ROOT / "industry_features"
BRANDS = ["cochlear", "medel", "ab"]

WIDEBAND_CKPT = ROOT / "best_wideband_tcn_512.pth"
COCHLEAR_CKPT = ROOT / "best_cochlear_tcn_512.pth"
MEDEL_CKPT = ROOT / "best_medel_tcn_512.pth"
AB_CKPT = ROOT / "best_ab_tcn_512.pth"

MAX_FRAMES = 1500
FRAME_HOP = 160
MAX_AUDIO_LEN = MAX_FRAMES * FRAME_HOP
SAMPLE_RATE = 16000

DEVICE = torch_directml.device()
print(f"Evaluating VAD models on {DEVICE}")



class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size,
                               padding=self.pad, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size,
                               padding=self.pad, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x):
        res = x
        out = self.conv1(x)
        out = out[:, :, :-self.pad]
        out = self.relu(self.bn1(out))
        out = self.conv2(out)
        out = out[:, :, :-self.pad]
        out = self.bn2(out)
        if self.downsample is not None:
            res = self.downsample(x)
        return self.relu(out + res)


class WidebandLongTCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 24, kernel_size=160, stride=160),
            ResidualBlock(24, 24, 3, 1),
            ResidualBlock(24, 24, 3, 2),
            ResidualBlock(24, 24, 3, 4),
            ResidualBlock(24, 24, 3, 8),
            ResidualBlock(24, 24, 3, 16),
            ResidualBlock(24, 24, 3, 32),
            ResidualBlock(24, 24, 3, 64),
            ResidualBlock(24, 24, 3, 128),
            ResidualBlock(24, 24, 3, 256),
            ResidualBlock(24, 24, 3, 512),
            nn.Conv1d(24, 1, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


class IndustryTCN(nn.Module):
    def __init__(self, input_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(input_channels, 24, 1),
            ResidualBlock(24, 24, 3, 1),
            ResidualBlock(24, 24, 3, 2),
            ResidualBlock(24, 24, 3, 4),
            ResidualBlock(24, 24, 3, 8),
            ResidualBlock(24, 24, 3, 16),
            ResidualBlock(24, 24, 3, 32),
            ResidualBlock(24, 24, 3, 64),
            ResidualBlock(24, 24, 3, 128),
            ResidualBlock(24, 24, 3, 256),
            ResidualBlock(24, 24, 3, 512),            
            nn.Conv1d(24, 1, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


def compute_metrics(y_true, y_score, threshold=0.5):
    n = min(len(y_true), len(y_score))
    y_true = np.asarray(y_true[:n]).astype(int)
    y_score = np.asarray(y_score[:n])

    y_pred = (y_score >= threshold).astype(int)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    total = len(y_true)
    acc = (tp + tn) / total if total > 0 else 0.0
    prec = tp / (tp + fp + 1e-12)
    rec = tp / (tp + fn + 1e-12)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    far = fp / (fp + tn + 1e-12)
    miss = fn / (tp + fn + 1e-12)

    try:
        auc = roc_auc_score(y_true, y_score)
    except Exception:
        auc = float("nan")

    return {
        "acc": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "far": far,
        "miss": miss,
        "auc": auc,
        "n_frames": int(total),
    }


def load_audio_16k_padded(path: Path):
    audio, sr = sf.read(str(path))
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        raise ValueError(f"Expected 16kHz, got {sr} in {path}")
    if len(audio) < MAX_AUDIO_LEN:
        audio = np.pad(audio, (0, MAX_AUDIO_LEN - len(audio)), mode='constant')
    else:
        audio = audio[:MAX_AUDIO_LEN]
    return audio


def main():
    # Load models
    wideband_model = WidebandLongTCN().to(DEVICE)
    wideband_model.load_state_dict(torch.load(WIDEBAND_CKPT, map_location=DEVICE))
    wideband_model.eval()

    cochlear_model = IndustryTCN(22).to(DEVICE)
    cochlear_model.load_state_dict(torch.load(COCHLEAR_CKPT, map_location=DEVICE))
    cochlear_model.eval()

    medel_model = IndustryTCN(12).to(DEVICE)
    medel_model.load_state_dict(torch.load(MEDEL_CKPT, map_location=DEVICE))
    medel_model.eval()

    ab_model = IndustryTCN(16).to(DEVICE)
    ab_model.load_state_dict(torch.load(AB_CKPT, map_location=DEVICE))
    ab_model.eval()

    stats = defaultdict(list) 

  
    label_files = sorted(LABEL_DIR.glob("*.npy"))
    print(f"Found {len(label_files)} test utterances for wideband/CI.")

    for lbl_path in tqdm(label_files, desc="Evaluating test"):
        utt_id = lbl_path.stem  
        noise_type = "mixed"
        snr_db = "all"

        y_true = np.load(lbl_path)
        if len(y_true) > MAX_FRAMES:
            y_true = y_true[:MAX_FRAMES]
        else:
            y_true = np.pad(y_true, (0, MAX_FRAMES - len(y_true)), mode='constant')

      
        audio_path = WIDEBAND_AUDIO_DIR / f"{utt_id}.wav"
        if not audio_path.exists():
            continue
        audio = load_audio_16k_padded(audio_path)

        X_wb = torch.tensor(audio, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits_wb = wideband_model(X_wb).cpu().numpy().flatten()
        logits_wb = logits_wb[:MAX_FRAMES]
        stats[("wideband", noise_type, snr_db)].append((y_true, logits_wb))

        for brand, model, ch_in in [
            ("cochlear", cochlear_model, 22),
            ("medel", medel_model, 12),
            ("ab", ab_model, 16),
        ]:
            feat_dir = INDUSTRY_FEATURE_ROOT / brand / "noisy_test" / "features"
            feat_path = feat_dir / f"{utt_id}.npy"
            if not feat_path.exists():
                continue
            feat = np.load(feat_path)  
            C, T = feat.shape
            if T < MAX_FRAMES:
                pad = MAX_FRAMES - T
                feat = np.pad(feat, ((0, 0), (0, pad)), mode='constant')
            else:
                feat = feat[:, :MAX_FRAMES]

            X_ci = torch.tensor(feat, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                logits_ci = model(X_ci).cpu().numpy().flatten()
            logits_ci = logits_ci[:MAX_FRAMES]
            stats[(brand, noise_type, snr_db)].append((y_true, logits_ci))

   
    out_csv = ROOT / "vad_test_metrics.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "model", "noise_type", "snr_db",
            "n_frames",
            "accuracy", "precision", "recall", "f1",
            "far", "miss", "auc"
        ])

        for key, pairs in stats.items():
            model_name, noise_type, snr_db = key
            all_true = np.concatenate([p[0] for p in pairs])
            all_score = np.concatenate([p[1] for p in pairs])
            m = compute_metrics(all_true, all_score)
            writer.writerow([
                model_name,
                noise_type,
                snr_db,
                m["n_frames"],
                f"{m['acc']:.4f}",
                f"{m['precision']:.4f}",
                f"{m['recall']:.4f}",
                f"{m['f1']:.4f}",
                f"{m['far']:.4f}",
                f"{m['miss']:.4f}",
                f"{m['auc']:.4f}" if not np.isnan(m["auc"]) else "nan",
            ])

    print(f"\nEvaluation complete. Metrics saved to: {out_csv}")


if __name__ == "__main__":
    main()


In [ ]:
import os
import csv
from pathlib import Path
from collections import defaultdict

import numpy as np
import soundfile as sf
from tqdm import tqdm

import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

import torch_directml


ROOT = Path(r"./TIMIT")

NOISY_TEST_MANIFEST = ROOT / "noisy_manifests" / "test_noisy_manifest.csv"
LABEL_DIR = ROOT / "frame_labels"

WIDEBAND_CKPT = ROOT / "best_wideband_tcn_512.pth"
COCHLEAR_CKPT = ROOT / "best_cochlear_tcn_512.pth"
MEDEL_CKPT = ROOT / "best_medel_tcn_512.pth"
AB_CKPT = ROOT / "best_ab_tcn_512.pth"

INDUSTRY_FEATURE_ROOT = ROOT / "industry_features"

MAX_FRAMES = 1500
FRAME_HOP = 160
MAX_AUDIO_LEN = MAX_FRAMES * FRAME_HOP
SAMPLE_RATE = 16000

DEVICE = torch_directml.device()
print(f"Evaluating detailed metrics on {DEVICE}")


class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size,
                               padding=self.pad, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size,
                               padding=self.pad, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x):
        res = x
        out = self.conv1(x)
        out = out[:, :, :-self.pad]
        out = self.relu(self.bn1(out))
        out = self.conv2(out)
        out = out[:, :, :-self.pad]
        out = self.bn2(out)
        if self.downsample is not None:
            res = self.downsample(x)
        return self.relu(out + res)


class WidebandLongTCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 24, kernel_size=160, stride=160),
            ResidualBlock(24, 24, 3, 1),
            ResidualBlock(24, 24, 3, 2),
            ResidualBlock(24, 24, 3, 4),
            ResidualBlock(24, 24, 3, 8),
            ResidualBlock(24, 24, 3, 16),
            ResidualBlock(24, 24, 3, 32),
            ResidualBlock(24, 24, 3, 64),
            ResidualBlock(24, 24, 3, 128),
            ResidualBlock(24, 24, 3, 256),
            ResidualBlock(24, 24, 3, 512),
            nn.Conv1d(24, 1, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


class IndustryTCN(nn.Module):
    def __init__(self, input_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(input_channels, 24, 1),
            ResidualBlock(24, 24, 3, 1),
            ResidualBlock(24, 24, 3, 2),
            ResidualBlock(24, 24, 3, 4),
            ResidualBlock(24, 24, 3, 8),
            ResidualBlock(24, 24, 3, 16),
            ResidualBlock(24, 24, 3, 32),
            ResidualBlock(24, 24, 3, 64),
            ResidualBlock(24, 24, 3, 128),
            ResidualBlock(24, 24, 3, 256),
            ResidualBlock(24, 24, 3, 512),
            nn.Conv1d(24, 1, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


try:
    torch.set_num_threads(1)
    silero_model, silero_utils = torch.hub.load(
        repo_or_dir='snakers4/silero-vad',
        model='silero_vad',
        force_reload=False
    )
    silero_model.eval()
    (get_speech_timestamps,
     _,  
     _,  
     _,  
     collect_chunks) = silero_utils
    HAS_SILERO = True
    print("Silero VAD loaded.")
except Exception as e:
    print(f"Could not load Silero VAD: {e}")
    HAS_SILERO = False



def compute_metrics(y_true, y_score, threshold=0.5):
    n = min(len(y_true), len(y_score))
    y_true = np.asarray(y_true[:n]).astype(int)
    y_score = np.asarray(y_score[:n])

    y_pred = (y_score >= threshold).astype(int)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    total = len(y_true)
    acc = (tp + tn) / total if total > 0 else 0.0
    prec = tp / (tp + fp + 1e-12)
    rec = tp / (tp + fn + 1e-12)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    far = fp / (fp + tn + 1e-12)
    miss = fn / (tp + fn + 1e-12)

    try:
        auc = roc_auc_score(y_true, y_score)
    except Exception:
        auc = float("nan")

    return {
        "acc": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "far": far,
        "miss": miss,
        "auc": auc,
        "n_frames": int(total),
    }


def load_audio_padded(path: Path):
    audio, sr = sf.read(str(path))
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        raise ValueError(f"Expected 16kHz, got {sr} in {path}")
    if len(audio) < MAX_AUDIO_LEN:
        audio = np.pad(audio, (0, MAX_AUDIO_LEN - len(audio)), mode='constant')
    else:
        audio = audio[:MAX_AUDIO_LEN]
    return audio


def silero_frame_scores(audio, frame_hop=FRAME_HOP):
    if not HAS_SILERO:
        return None
    wav_16k = torch.from_numpy(audio).float()
    with torch.no_grad():
        speech_ts = get_speech_timestamps(wav_16k, silero_model, sampling_rate=SAMPLE_RATE)
    n_frames = MAX_FRAMES
    scores = np.zeros(n_frames, dtype=float)
    for seg in speech_ts:
        start_samp = seg["start"]
        end_samp = seg["end"]
        start_frame = start_samp // frame_hop
        end_frame = end_samp // frame_hop
        start_frame = max(0, min(n_frames, start_frame))
        end_frame = max(0, min(n_frames, end_frame))
        scores[start_frame:end_frame] = 1.0
    return scores



def main():
    
    wideband_model = WidebandLongTCN().to(DEVICE)
    wideband_model.load_state_dict(torch.load(WIDEBAND_CKPT, map_location=DEVICE))
    wideband_model.eval()

    cochlear_model = IndustryTCN(22).to(DEVICE)
    cochlear_model.load_state_dict(torch.load(COCHLEAR_CKPT, map_location=DEVICE))
    cochlear_model.eval()

    medel_model = IndustryTCN(12).to(DEVICE)
    medel_model.load_state_dict(torch.load(MEDEL_CKPT, map_location=DEVICE))
    medel_model.eval()

    ab_model = IndustryTCN(16).to(DEVICE)
    ab_model.load_state_dict(torch.load(AB_CKPT, map_location=DEVICE))
    ab_model.eval()

    stats = defaultdict(list)

    if not NOISY_TEST_MANIFEST.exists():
        print(f"ERROR: manifest not found at {NOISY_TEST_MANIFEST}")
        return

    with open(NOISY_TEST_MANIFEST, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    print(f"[DEBUG] Manifest rows: {len(rows)}")
    if rows:
        print("[DEBUG] First row keys:", rows[0].keys())
        print("[DEBUG] First row example:", rows[0])

    processed_rows = 0
    skipped_missing_wav = 0
    skipped_missing_label = 0
    skipped_missing_feat = 0

    for row in tqdm(rows, desc="Noisy test"):
        utt_id = row.get("utt_id", "")
        noisy_path = Path(row.get("noisy_wav_path", ""))
        noise_type = row.get("noise_type", "unknown")
        snr_db = row.get("snr_db", "unknown")
        label_path = LABEL_DIR / f"{utt_id}_labels.npy"

        if not noisy_path.exists():
            skipped_missing_wav += 1
            
            continue
        if not label_path.exists():
            skipped_missing_label += 1
            
            continue

        y_true = np.load(label_path)
        if len(y_true) > MAX_FRAMES:
            y_true = y_true[:MAX_FRAMES]
        else:
            y_true = np.pad(y_true, (0, MAX_FRAMES - len(y_true)), mode='constant')

        audio = load_audio_padded(noisy_path)

        
        X_wb = torch.tensor(audio, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits_wb = wideband_model(X_wb).cpu().numpy().flatten()
        logits_wb = logits_wb[:MAX_FRAMES]
        stats[("wideband", noise_type, snr_db)].append((y_true, logits_wb))

        
        if HAS_SILERO:
            sil_scores = silero_frame_scores(audio)
            if sil_scores is not None:
                stats[("silero", noise_type, snr_db)].append((y_true, sil_scores))

        
        for brand, model, ch_in in [
            ("cochlear", cochlear_model, 22),
            ("medel", medel_model, 12),
            ("ab", ab_model, 16),
        ]:
            feat_dir = INDUSTRY_FEATURE_ROOT / brand / "noisy_test" / "features"
            feat_path = feat_dir / f"{utt_id}.npy"
            if not feat_path.exists():
                skipped_missing_feat += 1
                
                continue
            feat = np.load(feat_path)
            C, T = feat.shape
            if T < MAX_FRAMES:
                pad = MAX_FRAMES - T
                feat = np.pad(feat, ((0, 0), (0, pad)), mode='constant')
            else:
                feat = feat[:, :MAX_FRAMES]
            X_ci = torch.tensor(feat, dtype=torch.float32).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                logits_ci = model(X_ci).cpu().numpy().flatten()
            logits_ci = logits_ci[:MAX_FRAMES]
            stats[(brand, noise_type, snr_db)].append((y_true, logits_ci))

        processed_rows += 1

    print(f"[DEBUG] processed_rows: {processed_rows}")
    print(f"[DEBUG] skipped_missing_wav: {skipped_missing_wav}")
    print(f"[DEBUG] skipped_missing_label: {skipped_missing_label}")
    print(f"[DEBUG] skipped_missing_feat (CI): {skipped_missing_feat}")
    print(f"[DEBUG] total stats keys: {len(stats)}")
    for k, v in stats.items():
        print(f"[DEBUG] key {k}: {len(v)} utterances")

    out_csv = ROOT / "vad_test_metrics_detailed.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "model", "noise_type", "snr_db",
            "n_frames",
            "accuracy", "precision", "recall", "f1",
            "far", "miss", "auc"
        ])
        for key, pairs in stats.items():
            model_name, noise_type, snr_db = key
            all_true = np.concatenate([p[0] for p in pairs])
            all_score = np.concatenate([p[1] for p in pairs])
            m = compute_metrics(all_true, all_score)
            writer.writerow([
                model_name,
                noise_type,
                snr_db,
                m["n_frames"],
                f"{m['acc']:.4f}",
                f"{m['precision']:.4f}",
                f"{m['recall']:.4f}",
                f"{m['f1']:.4f}",
                f"{m['far']:.4f}",
                f"{m['miss']:.4f}",
                f"{m['auc']:.4f}" if not np.isnan(m["auc"]) else "nan",
            ])

    print(f"\nDetailed evaluation complete. Metrics saved to: {out_csv}")


if __name__ == "__main__":
    main()
